In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# CONFIGURATION
# ==========================================
FILE_PATH = 'master_catalog.csv'
OUTPUT_DIR = 'grid_search_plots'

os.makedirs(OUTPUT_DIR, exist_ok=True)
sns.set_theme(style="whitegrid", palette="muted")

# ==========================================
# DATA LOADING & CLEANING
# ==========================================
print("Loading catalog data...")

if not os.path.exists(FILE_PATH):
    print(f"ERROR: Could not find '{FILE_PATH}'. Have you run the training script yet?")
    sys.exit()

df = pd.read_csv(FILE_PATH)
print(f"Total rows in CSV (including failed runs): {len(df)}")

# Drop completely failed runs (where validation loss is NaN)
df = df.dropna(subset=['Best_Val_Loss'])

if len(df) == 0:
    print("ERROR: No valid runs found in the CSV. All recorded runs have NaN or missing validation losses.")
    print("Hint: Check your training script. If models are diverging instantly, lower your maximum learning rate.")
    sys.exit()

# Filter out the top 5% worst runs to keep plots readable
loss_cap = df['Best_Val_Loss'].quantile(0.95)
# df_clean = df[df['Best_Val_Loss'] <= loss_cap].copy()
df_clean = df.copy()  # Comment out outlier removal for now to see all data points

if len(df_clean) == 0:
    print("ERROR: df_clean is empty after filtering outliers.")
    sys.exit()

# Add logarithmic columns for better visual scaling
df_clean['Log10_Val_Loss'] = np.log10(df_clean['Best_Val_Loss'])

# Convert variables to strings/categories for grouping in plots
df_clean['LR_Cat'] = df_clean['LR'].astype(str)
df_clean['WD_Cat'] = df_clean['Weight_Decay'].astype(str)
df_clean['Num_Layers_Cat'] = df_clean['Num_Layers'].astype(str)
df_clean['P_Cat'] = df_clean['p'].astype(str)

print(f"Runs plotted (after outlier removal): {len(df_clean)}")

# ==========================================
# PLOT 1: Overall Loss Distribution
# ==========================================
plt.figure(figsize=(10, 6))
sns.histplot(df_clean['Log10_Val_Loss'], kde=True, bins=30, color='blue')
plt.title('Distribution of Best Validation Loss (Log10 Scale)')
plt.xlabel('Log10(Best Validation Loss)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_loss_distribution.png", dpi=300)
plt.close()

# ==========================================
# PLOT 2: Hyperparameter Boxplots
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Validation Loss vs. Individual Hyperparameters', fontsize=16)

sns.boxplot(ax=axes[0, 0], x='LR_Cat', y='Log10_Val_Loss', data=df_clean, order=sorted(df_clean['LR_Cat'].unique()))
axes[0, 0].set_title('Learning Rate')

sns.boxplot(ax=axes[0, 1], x='Num_Layers_Cat', y='Log10_Val_Loss', data=df_clean, order=sorted(df_clean['Num_Layers_Cat'].unique()))
axes[0, 1].set_title('Number of Layers')

sns.boxplot(ax=axes[1, 0], x='P_Cat', y='Log10_Val_Loss', data=df_clean, order=sorted(df_clean['P_Cat'].unique()))
axes[1, 0].set_title('Latent Dimension (p)')

sns.boxplot(ax=axes[1, 1], x='WD_Cat', y='Log10_Val_Loss', data=df_clean, order=sorted(df_clean['WD_Cat'].unique()))
axes[1, 1].set_title('Weight Decay')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_hyperparameter_boxplots.png", dpi=300)
plt.close()

# ==========================================
# PLOT 3: The Interaction Heatmap (LR vs Layers)
# ==========================================
pivot_lr_layers = df_clean.pivot_table(
    values='Log10_Val_Loss', 
    index='Num_Layers', 
    columns='LR_Cat', 
    aggfunc=np.median
)

plt.figure(figsize=(10, 8))
sns.heatmap(pivot_lr_layers, annot=True, cmap='viridis_r', fmt=".3f")
plt.title('Median Log10(Val Loss) - Learning Rate vs. Num Layers')
plt.xlabel('Learning Rate')
plt.ylabel('Number of Layers')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/03_heatmap_lr_vs_layers.png", dpi=300)
plt.close()

# ==========================================
# PLOT 4: The Interaction Heatmap (p vs Weight Decay)
# ==========================================
pivot_p_wd = df_clean.pivot_table(
    values='Log10_Val_Loss', 
    index='P_Cat', 
    columns='WD_Cat', 
    aggfunc=np.median
)

plt.figure(figsize=(10, 8))
sns.heatmap(pivot_p_wd, annot=True, cmap='viridis_r', fmt=".3f")
plt.title('Median Log10(Val Loss) - Latent Dim (p) vs. Weight Decay')
plt.xlabel('Weight Decay')
plt.ylabel('Latent Dimension (p)')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/04_heatmap_p_vs_wd.png", dpi=300)
plt.close()

# ==========================================
# PLOT 5: Global Pairplot
# ==========================================
pairplot_vars = ['Log10_Val_Loss', 'Epochs_Trained', 'Num_Layers', 'p']
pp = sns.pairplot(df_clean[pairplot_vars], diag_kind='kde', corner=True, height=2.5)
pp.fig.suptitle('Global Numerical Interactions', y=1.02, fontsize=16)
plt.savefig(f"{OUTPUT_DIR}/05_global_pairplot.png", dpi=300)
plt.close()

# ==========================================
# PLOT 6: Top 10 Best Configurations
# ==========================================
top_10 = df_clean.sort_values(by='Best_Val_Loss', ascending=True).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x='Best_Val_Loss', y='Run_Name', data=top_10, palette='crest')
plt.title('Top 10 Best Hyperparameter Configurations')
plt.xlabel('Best Validation Loss (Lower is Better)')
plt.ylabel('Configuration ID')
plt.xscale('log') 
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/06_top_10_configurations.png", dpi=300)
plt.close()

print(f"Success. Plots have been generated and saved to the '{OUTPUT_DIR}' directory.")

Loading catalog data...
Total rows in CSV (including failed runs): 74
Runs plotted (after outlier removal): 74


c:\Users\Lohith\miniconda3\envs\torch\lib\site-packages\numpy\lib\function_base.py:4573: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\Lohith\miniconda3\envs\torch\lib\site-packages\numpy\lib\function_base.py:4573: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


Success. Plots have been generated and saved to the 'grid_search_plots' directory.
